In [1]:
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
from torch.optim import Adam
import matplotlib.pyplot as plt

# Confirm torch version and device
print(f'PyTorch version: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

PyTorch version: 2.9.1+cu128
Device: cuda


In [2]:
# Features and targets
X_norm       = np.load('data/X_norm_clean.npy')
R            = np.load('data/spec5refl_R.npy')
E            = np.load('data/endmem21_E.npy')
valid_mask   = np.load('data/spectral_valid_mask.npy')

# Spatial Laplacians
L_elev   = sp.load_npz('data/L_elev.npz')
L_aspect = sp.load_npz('data/L_aspect.npz')
L_ddem   = sp.load_npz('data/L_ddem.npz')

# Spectral Laplacian
L_band = np.load('data/L_band.npy')

print(f'X_norm       : {X_norm.shape}   dtype={X_norm.dtype}')
print(f'R            : {R.shape}   dtype={R.dtype}')
print(f'E            : {E.shape}   dtype={E.dtype}')
print(f'valid_mask   : {valid_mask.shape}  dtype={valid_mask.dtype}')
print(f'L_elev       : {L_elev.shape}  nnz={L_elev.nnz:,}')
print(f'L_aspect     : {L_aspect.shape}  nnz={L_aspect.nnz:,}')
print(f'L_ddem       : {L_ddem.shape}  nnz={L_ddem.nnz:,}')
print(f'L_band       : {L_band.shape}')

X_norm       : (2041172, 12)   dtype=float32
R            : (2041172, 5)   dtype=float32
E            : (21, 5)   dtype=float32
valid_mask   : (2041172,)  dtype=bool
L_elev       : (2041172, 2041172)  nnz=18,345,196
L_aspect     : (2041172, 2041172)  nnz=18,347,260
L_ddem       : (2041172, 2041172)  nnz=18,267,646
L_band       : (5, 5)


# Rebuild normalized feature matrix dropping log_accum (topo col index 5)
# X_norm columns:
#   0-4  : spectral (Blue, Green, Red, RedEdge, NIR)
#   5-10 : topo (elevation, slope, asin, acos, curvature, log_accum)
#   11   : panchromatic
#   12   : LWIR
# Drop col 10 (log_accum)

keep_cols = list(range(10)) + [11, 12]   # drop index 10
X_norm_clean = X_norm[:, keep_cols]

print(f'X_norm_clean shape: {X_norm_clean.shape}  (dropped log_accum)')
print()

# Verify log_accum is gone and no extreme values remain
col_names_clean = ['Blue', 'Green', 'Red', 'RedEdge', 'NIR',
                   'elevation', 'slope', 'aspect_sin', 'aspect_cos',
                   'curvature', 'panchromatic', 'lwir']
print(f'{"Feature":>14}  {"Min":>12}  {"Max":>12}  {"NaNs":>8}')
print('-' * 54)
for i, name in enumerate(col_names_clean):
    col = X_norm_clean[:, i]
    nans = np.isnan(col).sum()
    valid = col[~np.isnan(col)]
    print(f'{name:>14}  {valid.min():>12.4f}  {valid.max():>12.4f}  {nans:>8,}')

print()

# Zero out NaN rows (boundary fringe pixels)
# These are excluded from reconstruction loss by mask anyway
nan_rows = np.isnan(X_norm_clean).any(axis=1)
X_norm_clean[nan_rows] = 0.0
print(f'Zeroed {nan_rows.sum():,} NaN rows')
print(f'Remaining NaNs: {np.isnan(X_norm_clean).sum()}')

# Save updated feature matrix
np.save('data/X_norm_clean.npy', X_norm_clean)
print('Saved data/X_norm_clean.npy')

In [3]:
# GPU memory available
if torch.cuda.is_available():
    gpu_mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_mem_free  = torch.cuda.mem_get_info()[0] / 1e9
    print(f'GPU: {torch.cuda.get_device_properties(0).name}')
    print(f'  Total memory : {gpu_mem_total:.1f} GB')
    print(f'  Free memory  : {gpu_mem_free:.1f} GB')
    print()

# Estimate memory footprint of key arrays
def mem_mb(arr):
    return arr.nbytes / 1e6

print('CPU array sizes:')
print(f'  X_norm         : {mem_mb(X_norm):.1f} MB')
print(f'  R              : {mem_mb(R):.1f} MB')
print(f'  E              : {mem_mb(E):.3f} MB')
print()

# Sparse Laplacian memory (data + indices)
def sparse_mem_mb(L):
    return (L.data.nbytes + L.indices.nbytes + L.indptr.nbytes) / 1e6

print('Sparse Laplacian sizes (CSR):')
print(f'  L_elev         : {sparse_mem_mb(L_elev):.1f} MB')
print(f'  L_aspect       : {sparse_mem_mb(L_aspect):.1f} MB')
print(f'  L_ddem         : {sparse_mem_mb(L_ddem):.1f} MB')
print()

# Estimate A matrix (memberships) size
N, K = X_norm.shape[0], E.shape[0]
A_size_mb = N * K * 4 / 1e6   # float32
print(f'A matrix (N={N}, K={K}): {A_size_mb:.1f} MB')
print()
print(f'Estimated total GPU requirement: '
      f'{mem_mb(X_norm) + mem_mb(R) + 3*sparse_mem_mb(L_elev) + A_size_mb:.1f} MB')

GPU: NVIDIA RTX PRO 2000 Blackwell Generation Laptop GPU
  Total memory : 8.5 GB
  Free memory  : 7.3 GB

CPU array sizes:
  X_norm         : 98.0 MB
  R              : 40.8 MB
  E              : 0.000 MB

Sparse Laplacian sizes (CSR):
  L_elev         : 228.3 MB
  L_aspect       : 228.3 MB
  L_ddem         : 227.4 MB

A matrix (N=2041172, K=21): 171.5 MB

Estimated total GPU requirement: 995.2 MB


In [4]:
# convert dense arrays to torch tensors on GPU
X      = torch.tensor(X_norm, dtype=torch.float32, device=device)
R_tens = torch.tensor(R,      dtype=torch.float32, device=device)
E_tens = torch.tensor(E,       dtype=torch.float32, device=device)
mask   = torch.tensor(valid_mask, dtype=torch.bool, device=device)
L_band_tens = torch.tensor(L_band, dtype=torch.float32, device=device)

print(f'X      : {X.shape}  {X.device}')
print(f'R      : {R_tens.shape}  {R_tens.device}')
print(f'E      : {E_tens.shape}  {E_tens.device}')
print(f'mask   : {mask.shape}  {mask.device}')
print(f'L_band : {L_band_tens.shape}  {L_band_tens.device}')
print()

X      : torch.Size([2041172, 12])  cuda:0
R      : torch.Size([2041172, 5])  cuda:0
E      : torch.Size([21, 5])  cuda:0
mask   : torch.Size([2041172])  cuda:0
L_band : torch.Size([5, 5])  cuda:0



In [5]:
# convert sparse scipy CSR matrices to torch sparse CSR tensors on GPU
def scipy_csr_to_torch_sparse(L, device):
    """convert scipy csr sparse matrix to torch sparse csr tensor"""
    L = L.astype(np.float32)
    crow_indices = torch.tensor(L.indptr,  dtype=torch.int32)
    col_indices  = torch.tensor(L.indices, dtype=torch.int32)
    values       = torch.tensor(L.data,    dtype=torch.float32)
    size         = L.shape
    return torch.sparse_csr_tensor(
        crow_indices, col_indices, values,
        size=size, dtype=torch.float32
    ).to(device)

In [6]:
print("converting spatial laplacians to torch sparse csr...")
L_elev_t = scipy_csr_to_torch_sparse(L_elev, device)
L_aspect_t = scipy_csr_to_torch_sparse(L_aspect, device)
L_ddem_t = scipy_csr_to_torch_sparse(L_ddem, device)

print(f'L_elev   : {L_elev_t.shape} {L_elev_t.device}')
print(f'L_aspect : {L_aspect_t.shape} {L_aspect_t.device}')
print(f'L_ddem_t : {L_ddem_t.shape} {L_ddem_t.device}')

# report GPU memory after loading 
mem_used = (torch.cuda.memory_allocated() / 1e9)
mem_free = (torch.cuda.mem_get_info()[0] / 1e9)
print(f'gpu memory allocated : {mem_used:.2f} GB')
print(f'gpu memory free      : {mem_free:.2f} GB')

converting spatial laplacians to torch sparse csr...


/tmp/ipykernel_17771/1075959278.py:9: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  return torch.sparse_csr_tensor(


L_elev   : torch.Size([2041172, 2041172]) cuda:0
L_aspect : torch.Size([2041172, 2041172]) cuda:0
L_ddem_t : torch.Size([2041172, 2041172]) cuda:0
gpu memory allocated : 0.61 GB
gpu memory free      : 6.66 GB


In [7]:
# CPD (canonical polyadic decomposition) Tensor Layer
class CPDTensorLayer(nn.Module):
    """
    rank-R canonical polyadic decomposition (CPD) tensor layer
    mapping (spectral + topo) -> hidden

    Input:
        x_i ∈ ℝ^{D_in} split into:
            s_i ∈ ℝ^{D_s} (spectral, first D_s features (here 5 bands))
            t_i ∈ ℝ^{D_t} (topographic+aux, remaining features (here 6 topo + 2 aux bands)

    Weight tensor:
        W ∈ ℝ^{D_s x D_t x D_out} in rank-R CPD format:
        W = Σ_r (u_r^(1) ⊗ u_r^(2) ⊗ u_r^(3))

        where ⊗ := outer product
        and u_r^(*) is the r-th decomposition vector of U(*)

    Factor matrices:
        U1 ∈ ℝ^{D_s  x R}   spectral mode -- regularized by L_band
        U2 ∈ ℝ^{D_t  x R}   topographic mode 
        U3 ∈ ℝ^{D_out  x R} output mode

        'mode' in the tensor sense
        R (hyperparameter) CPD rank: 
            number of rank-1 terms in the decomposition
            controls how expressive the weight tensor is
        D_out (hyperparameter) output dimension of the tensor layer:
            determines dimensionality of the hidden space
            that passes to the attention layer
        

    Forward pass -- (tensor) efficient: never builds full W:
        h_i = Σ_r (s_i · u_r^(1)) * (t_i · u_r^(2)) * u_r^(3)
            = U3 · ((U1ᵀ s_i) * (U2ᵀ t_i))
            outputs a vector of length D_out
            note: 1st line '·' = vector inner product
                           '*' = scalar multiplication
                  2nd line '.' = matrix-vector product
                           '*' = element-wise vector multiplication
        
    """
    
    def __init__(self, D_s, D_t, D_out, R):
        super().__init__()
        self.D_s = D_s
        self.D_t = D_t
        self.R   = R

        # CPD factor matrices
        self.U1 = nn.Parameter(torch.empty(D_s,   R))
        self.U2 = nn.Parameter(torch.empty(D_t,   R))
        self.U3 = nn.Parameter(torch.empty(D_out, R))

        #initialize with small random values
        nn.init.xavier_normal_(self.U1)
        nn.init.xavier_normal_(self.U2)
        nn.init.xavier_normal_(self.U3)

        self.norm = nn.LayerNorm(D_out)


    def forward(self, x):
        """
        x : (N, D_s + D_t)
        returns h : (N, D_out)
        """
        s = x[:, :self.D_s]        # (N, D_s)
        t = x[:, self.D_s:]        # (N, D_t)

        # project each mode onto its factor matrix
        # (N, D_s) @ (D_s, R) -> (N, R)
        alpha = s @ self.U1      # spectral projections
        # (N, D_t) @ (D_t, R) -> (N, R)
        beta  = t @ self.U2      # topo projections

        # element-wise product of mode projections
        # (N, R) * (N, R) -> (N, R)
        gamma = alpha * beta

        # project to output dimension
        # (N, R) @ (R, D_out) -> (N, D_out)
        h = gamma @ self.U3.T

        return torch.relu(self.norm(h))


    def band_regularization(self, L_band):
        """
        Graph Laplacian regularization on spectral factor matrix U1
        tr(U1ᵀ L_band U1) -- encourages spectrally similar bands
                            to have similar factor vectors
        L_band : (D_s, D_s) -- dense tensor
        """
        return torch.trace(self.U1.T @ L_band @ self.U1)

print('CPDTensorLayer defined')        

CPDTensorLayer defined


In [8]:
### cross-modal attention module ###
# graph-laplacian regularization approximates INTRA-modal attention
# by encouraging smoothness within a mode
## cross-modal attention is between the CPD hidden representation
# and the raw input features

class CrossModalAttention(nn.Module):
    """
    cross-modal attention between the CPD hidden representation
    and the raw input features

    Query   : from CPD hidden h ∈ ℝ^{D_hidden}
    Key     : from raw input  x ∈ ℝ^{D_in}
    Value   : from raw input  x ∈ ℝ^{D_in}

    Learns which input features are most informative
        given the joint spectral-topographic representation.
    Residual connection added for training stability
    """
    def __init__(self, D_hidden, D_in, D_attn):
        super().__init__()
        self.D_attn = D_attn
        self.scale  = D_attn ** -0.5

        self.W_Q = nn.Linear(D_hidden, D_attn, bias=False)
        self.W_K = nn.Linear(D_in,     D_attn, bias=False)
        self.W_V = nn.Linear(D_in,     D_attn, bias=False)

        # project attention output back to D_hidden for residual
        self.W_O = nn.Linear(D_attn, D_hidden, bias=False)

        self.norm = nn.LayerNorm(D_hidden)

    def forward(self, h, x):
        """
        h : (N, D_hidden) CPD layer output
        x : (N, D_in)     raw normalized input
        returns h' : (N, D_hidden)
        """
        Q = self.W_Q(h)        # (N, D_attn)
        K = self.W_K(x)        # (N, D_attn)
        V = self.W_V(x)        # (N, D_attn)

        # per-pixel self-attention score (scalar per pixel)
        # each pixel attends to its features only --- 
        # no cross-pixel attention (that would be O(N^2))
        # spatial structure is "attended" by the laplacians
        attn = torch.sum(Q * K, dim=1, keepdim=True) * self.scale
        attn = torch.sigmoid(attn)   # (N, 1)

        # weighted value
        out = attn * V          # (N, D_attn)
        out = self.W_O(out)     # (N, D_hidden)

        # residual + LayerNorm
        return self.norm(h + out)

print('CrossModalAttention defined')

CrossModalAttention defined


In [9]:
### FULL MODEL ###
class GRAFTN(nn.Module):
    """
    Graph-Regularized Attention Fuzzy Tensor Network

    Architecture:
        1. CPD Tensor Layer  : (spectral x topo) -> hidden
        2. Cross-Modal Attn  : refine hidden using raw input
        3. Membership Head   : hidden -> sigmoid -> [0,1]^k
        where k := number of endmembers to unmix into

    Loss (computed externally):
        reconstruction ||R - A·Eᵀ||²
        + spatial smoothness 
                λ^(*)·tr(AᵀL_(*) A) 
                where (*) ∈ {elev, aspect, ddem}
        + spectral factor smoothness
                λ^(band)·tr(AᵀL_(band) A)
    """

    def __init__(self,
                 D_s=5,     # spectral input dim
                 D_t=8,     # topo+aux input dim
                 D_out=32,  # CPD output (hidden) dim
                 R=8,       # CPD rank
                 D_attn=16, # attention dim
                 k=21):     # number of endmembers
        super().__init__()

        self.D_s = D_s
        self.D_t = D_t
        D_in = D_s + D_t

        # CPD tensor layer
        self.cpd = CPDTensorLayer(D_s, D_t, D_out, R)

        # Cross-modal attention
        self.attn = CrossModalAttention(D_out, D_in, D_attn)

        # Membership head
        self.head = nn.Sequential(
            nn.Linear(D_out, k),
            nn.Sigmoid()
        )

    def forward(self, x):
        """
        x : (N, D_s + D_t) normalized input features
        returns a : (N, k) fuzzy memberships in [0, 1]
        """
        h  = self.cpd(x)      # (N, D_out)
        h  = self.attn(h, x)  # (N, D_out)
        a  = self.head(h)     # (N, k)

        return a

    def band_regularization(self, L_band):
        return self.cpd.band_regularization(L_band)

print('GRAFTN defined')

GRAFTN defined


In [24]:
### build loss function ###
def graftn_loss(A, X, R_tens, E_tens, mask,
                L_elev_t, L_aspect_t, L_ddem_t,
                L_band_tens, model,
                lam_elev, lam_aspect, lam_ddem, lam_band):
    """
    GRAFTN loss function

    Terms:
        1. Spectral reconstruction  : ||R_valid - A_valid @ E^T||^2_F
        2. Elevation smoothness     : lambda_elev   * tr(A^T L_elev   A)
        3. Aspect smoothness        : lambda_aspect * tr(A^T L_aspect A)
        4. dDEM smoothness          : lambda_ddem   * tr(A^T L_ddem   A)
        5. Band factor smoothness   : lambda_band   * tr(U1^T L_band U1)

    Parameters
    ----------
    A          : (N, k) membership matrix, output of model
    X          : (N, 13) normalized input features
    R_tens     : (N, 5) raw reflectance (mosaic values)
    E_tens     : (k, 5) endmember matrix
    mask       : (N,) bool -- pixels with valid spectral values
    L_*_t      : (N, N) sparse torch CSR Laplacians (elev, aspect, ddem)
    L_band_tens: (5, 5) dense spectral Laplacian
    model      : GRAFTN instance (for band regularization)
    lam_*      : scalar regularization weights (lambda)

    Returns
    --------
    loss_total, loss_recon, loss_spatial, loss_band
    """

    # ----- 1. Spectral reconstruction loss ----
    # only over spectrally valid pixels
    A_valid = A[mask]          # (N_valid, k)
    R_valid = R_tens[mask]     # (N_valid, k)

    # Predicted reflectance: A_valid @ E^T shape (N_valid, 5)
    # Divide E by k so uniform memberships ~0.5 produce
    # predicted reflectance at correct magnitude
    k = E_tens.shape[0]
    R_pred = A_valid @ (E_tens / k)
    # R_pred = A_valid @ E_tens  # (N_valid, 5) # previous

    # Frobenius norm squared, normalized by number of valid pixels
    loss_recon = torch.sum((R_valid - R_pred) ** 2) / mask.sum()


    # ----- 2. Spatial smoothness losses ---
    # tr(A^T L A) = sum_ij w_ij(a_i - a_j)^2
    # compute efficiently as tr(A^T (L A))
    # L is sparse (N,N), A is dense (N,k)
    # torch.mm not supported for sparse @ dense in all versions
    # use torch.sparse.mm instead

    LA_elev   = torch.sparse.mm(L_elev_t,   A) # (N, k)
    LA_aspect = torch.sparse.mm(L_aspect_t, A) # (N, k)
    LA_ddem   = torch.sparse.mm(L_ddem_t,   A) # (N, k)

    # tr(A^T LA) = sum of element-wise product of A * LA
    loss_elev    = torch.sum(A * LA_elev)   / A.shape[0]
    loss_aspect  = torch.sum(A * LA_aspect) / A.shape[0]
    loss_ddem    = torch.sum(A * LA_ddem)   / A.shape[0]

    loss_spatial = (lam_elev   * loss_elev +
                    lam_aspect * loss_aspect +
                    lam_ddem   * loss_ddem)

    # ------ 3. Band factor regularization --- 
    loss_band = lam_band * model.band_regularization(L_band_tens)

    # ------ 4. Total ----
    loss_total = loss_recon + loss_spatial + loss_band

    return loss_total, loss_recon, loss_spatial, loss_band

print('graftn_loss defined')   

graftn_loss defined


In [27]:
# instantiate and move to device
model = GRAFTN(
    D_s=5, D_t=7, D_out=32, R=8, D_attn=16, k=21 # D_t = 7 -- removed log_acc
).to(device)

In [19]:
# parameter count
total_params = sum(p.numel() for p in model.parameters())
print(f'model parameters: {total_params:,}')
print()
for name, p in model.named_parameters():
    print(f' {name:<30} {str(p.shape):<25} {p.numel():>6,}')

model parameters: 2,581

 cpd.U1                         torch.Size([5, 8])            40
 cpd.U2                         torch.Size([7, 8])            56
 cpd.U3                         torch.Size([32, 8])          256
 cpd.norm.weight                torch.Size([32])              32
 cpd.norm.bias                  torch.Size([32])              32
 attn.W_Q.weight                torch.Size([16, 32])         512
 attn.W_K.weight                torch.Size([16, 12])         192
 attn.W_V.weight                torch.Size([16, 12])         192
 attn.W_O.weight                torch.Size([32, 16])         512
 attn.norm.weight               torch.Size([32])              32
 attn.norm.bias                 torch.Size([32])              32
 head.0.weight                  torch.Size([21, 32])         672
 head.0.bias                    torch.Size([21])              21


In [21]:
#### Training Loop ####

In [28]:
# hyperparameters 
LEARNING_RATE = 1e-3
N_EPOCHS      = 100
LOG_EVERY     = 10 # print loss every * epochs

# regularization weights
# reconstruction loss normalized by N_valid (~1.99M pixels)
# spatial losses normalized by N (~2.04M pixels)
# band loss is a small scalar -- lam_band can be larger
# using empirically derived values
LAM_ELEV    = 1.0
LAM_ASPECT  = 0.65
LAM_DDEM    = 0.78
LAM_BAND    = 0.014

optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

# training history
history = {
    'total':[], 'recon':[], 'spatial':[], 'band':[]
}

print('GRAFTN Training')
print(f'  Epochs        : {N_EPOCHS}')
print(f'  Learning rate : {LEARNING_RATE}')
print(f'  lam_elev      : {LAM_ELEV}')
print(f'  lam_aspect    : {LAM_ASPECT}')
print(f'  lam_ddem      : {LAM_DDEM}')
print(f'  lam_band      : {LAM_BAND}')
print()

import time
t_start = time.time()

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    # forward pass -- all N pixels at once
    A = model(X)   # (N, k)

    # loss 
    loss_total, loss_recon, loss_spatial, loss_band = graftn_loss(
        A, X, R_tens, E_tens, mask,
        L_elev_t, L_aspect_t, L_ddem_t,
        L_band_tens, model,
        LAM_ELEV, LAM_ASPECT, LAM_DDEM, LAM_BAND
    )

    # backward pass
    loss_total.backward()
    optimizer.step()

    # record history
    history['total'].append(loss_total.item())
    history['recon'].append(loss_recon.item())
    history['spatial'].append(loss_spatial.item())
    history['band'].append(loss_band.item())

    if epoch % LOG_EVERY == 0 or epoch == 1:
        elapsed = time.time() - t_start
        print(f'Epoch {epoch:>4d}/{N_EPOCHS}  '
              f'total={loss_total.item():.6f}  '
              f'recon={loss_recon.item():.6f}  '
              f'spatial={loss_spatial.item():.6f}  '
              f'band={loss_band.item():.6f}  '
              f'({elapsed:.1f}s)')

print()
print(f'training complete in {time.time()-t_start:.1f} s')

GRAFTN Training
  Epochs        : 100
  Learning rate : 0.001
  lam_elev      : 1.0
  lam_aspect    : 0.65
  lam_ddem      : 0.78
  lam_band      : 0.014

Epoch    1/100  total=0.446636  recon=0.116109  spatial=0.252082  band=0.078444  (17.1s)
Epoch   10/100  total=0.382524  recon=0.112305  spatial=0.194433  band=0.075786  (456.6s)
Epoch   20/100  total=0.336723  recon=0.107907  spatial=0.156060  band=0.072756  (791.1s)
Epoch   30/100  total=0.307551  recon=0.103887  spatial=0.133976  band=0.069689  (1145.8s)
Epoch   40/100  total=0.287724  recon=0.100456  spatial=0.120581  band=0.066686  (1507.4s)
Epoch   50/100  total=0.273386  recon=0.097548  spatial=0.112037  band=0.063802  (1869.6s)
Epoch   60/100  total=0.262111  recon=0.094952  spatial=0.106111  band=0.061048  (2232.7s)
Epoch   70/100  total=0.252630  recon=0.092519  spatial=0.101689  band=0.058423  (2598.4s)
Epoch   80/100  total=0.244563  recon=0.090249  spatial=0.098396  band=0.055918  (2959.5s)
Epoch   90/100  total=0.237661

In [14]:
model_test = GRAFTN(
    D_s=5, D_t=7, D_out=32, R=8, D_attn=16, k=21
).to(device)

with torch.no_grad():
    A_test = model_test(X)

print('A_test stats:')
print(f'  shape : {A_test.shape}')
print(f'  min   : {A_test.min().item():.6f}')
print(f'  max   : {A_test.max().item():.6f}')
print(f'  mean  : {A_test.mean().item():.6f}')
print(f'  NaNs  : {torch.isnan(A_test).sum().item()}')
print(f'  Infs  : {torch.isinf(A_test).sum().item()}')
print()

# Check each layer output individually
with torch.no_grad():
    # CPD layer
    h_cpd = model_test.cpd(X)
    print(f'CPD output:')
    print(f'  NaNs: {torch.isnan(h_cpd).sum().item()}')
    print(f'  Infs: {torch.isinf(h_cpd).sum().item()}')
    print(f'  min : {h_cpd.min().item():.4f}')
    print(f'  max : {h_cpd.max().item():.4f}')
    print()

    # Attention layer
    h_attn = model_test.attn(h_cpd, X)
    print(f'Attention output:')
    print(f'  NaNs: {torch.isnan(h_attn).sum().item()}')
    print(f'  Infs: {torch.isinf(h_attn).sum().item()}')
    print(f'  min : {h_attn.min().item():.4f}')
    print(f'  max : {h_attn.max().item():.4f}')
    print()

    # Membership head
    a_head = model_test.head(h_attn)
    print(f'Head output:')
    print(f'  NaNs: {torch.isnan(a_head).sum().item()}')
    print(f'  Infs: {torch.isinf(a_head).sum().item()}')
    print(f'  min : {a_head.min().item():.4f}')
    print(f'  max : {a_head.max().item():.4f}')
    print()

    # Reconstruction loss alone
    A_valid = A_test[mask]
    R_valid = R_tens[mask]
    R_pred  = A_valid @ E_tens
    loss_recon = torch.sum((R_valid - R_pred)**2) / mask.sum()
    print(f'Reconstruction loss: {loss_recon.item():.6f}')
    print(f'  R_pred NaNs: {torch.isnan(R_pred).sum().item()}')
    print()

    # Spatial loss -- check sparse mm output
    LA = torch.sparse.mm(L_elev_t, A_test)
    print(f'Sparse mm (L_elev @ A):')
    print(f'  NaNs: {torch.isnan(LA).sum().item()}')
    print(f'  Infs: {torch.isinf(LA).sum().item()}')
    print(f'  min : {LA.min().item():.4f}')
    print(f'  max : {LA.max().item():.4f}')

A_test stats:
  shape : torch.Size([2041172, 21])
  min   : 0.087444
  max   : 0.917582
  mean  : 0.493679
  NaNs  : 0
  Infs  : 0

CPD output:
  NaNs: 0
  Infs: 0
  min : 0.0000
  max : 3.5484

Attention output:
  NaNs: 0
  Infs: 0
  min : -2.5593
  max : 4.7689

Head output:
  NaNs: 0
  Infs: 0
  min : 0.0874
  max : 0.9176

Reconstruction loss: 46.757233
  R_pred NaNs: 0

Sparse mm (L_elev @ A):
  NaNs: 0
  Infs: 0
  min : -0.6457
  max : 0.8800


In [15]:
with torch.no_grad():
    s = X[:, :5]        # (N, 5)
    t = X[:, 5:]        # (N, 8)

    U1 = model_test.cpd.U1
    U2 = model_test.cpd.U2
    U3 = model_test.cpd.U3

    alpha = s @ U1      # (N, R)
    beta  = t @ U2      # (N, R)
    gamma = alpha * beta # (N, R)
    h_pre = gamma @ U3.T # (N, D_out)

    print('Pre-LayerNorm stats:')
    print(f'  NaNs : {torch.isnan(h_pre).sum().item()}')
    print(f'  Infs : {torch.isinf(h_pre).sum().item()}')
    print(f'  min  : {h_pre.min().item():.6f}')
    print(f'  max  : {h_pre.max().item():.6f}')
    print(f'  mean : {h_pre.mean().item():.6f}')
    print(f'  std  : {h_pre.std().item():.6f}')
    print()

    # Check variance per row -- LayerNorm divides by per-row std
    row_std = h_pre.std(dim=1)
    print(f'Per-row std stats:')
    print(f'  min  : {row_std.min().item():.8f}')
    print(f'  max  : {row_std.max().item():.8f}')
    print(f'  mean : {row_std.mean().item():.8f}')
    print(f'  zero rows: {(row_std == 0).sum().item()}')
    print(f'  near-zero rows (< 1e-5): {(row_std < 1e-5).sum().item()}')
    print()

    # Check intermediate steps
    print('Alpha stats:')
    print(f'  NaNs: {torch.isnan(alpha).sum().item()}')
    print(f'  min : {alpha.min().item():.6f}  max: {alpha.max().item():.6f}')
    print()
    print('Beta stats:')
    print(f'  NaNs: {torch.isnan(beta).sum().item()}')
    print(f'  min : {beta.min().item():.6f}  max: {beta.max().item():.6f}')
    print()
    print('Gamma stats:')
    print(f'  NaNs: {torch.isnan(gamma).sum().item()}')
    print(f'  min : {gamma.min().item():.6f}  max: {gamma.max().item():.6f}')

Pre-LayerNorm stats:
  NaNs : 0
  Infs : 0
  min  : -31.480423
  max  : 16.455154
  mean : 0.006312
  std  : 0.638937

Per-row std stats:
  min  : 0.00000000
  max  : 11.45674992
  mean : 0.49504891
  zero rows: 52926
  near-zero rows (< 1e-5): 52926

Alpha stats:
  NaNs: 0
  min : -8.583163  max: 6.135272

Beta stats:
  NaNs: 0
  min : -9.427881  max: 9.946370

Gamma stats:
  NaNs: 0
  min : -16.425812  max: 50.140598


In [21]:
with torch.no_grad():
    s = X[:, :5]
    t = X[:, 5:]

    print('s (spectral input) stats:')
    print(f'  NaNs: {torch.isnan(s).sum().item()}')
    print(f'  Infs: {torch.isinf(s).sum().item()}')
    print(f'  min : {s[~torch.isnan(s)].min().item():.6f}')
    print(f'  max : {s[~torch.isnan(s)].max().item():.6f}')
    print()

    print('t (topo input) stats:')
    print(f'  NaNs: {torch.isnan(t).sum().item()}')
    print(f'  Infs: {torch.isinf(t).sum().item()}')
    print(f'  min : {t[~torch.isnan(t)].min().item():.6f}')
    print(f'  max : {t[~torch.isnan(t)].max().item():.6f}')
    print()

    print('U1 stats:')
    print(f'  NaNs: {torch.isnan(model_test.cpd.U1).sum().item()}')
    print(f'  min : {model_test.cpd.U1.min().item():.6f}')
    print(f'  max : {model_test.cpd.U1.max().item():.6f}')
    print()

    print('U2 stats:')
    print(f'  NaNs: {torch.isnan(model_test.cpd.U2).sum().item()}')
    print(f'  min : {model_test.cpd.U2.min().item():.6f}')
    print(f'  max : {model_test.cpd.U2.max().item():.6f}')
    print()

    # Check which rows of X contain NaN
    nan_rows_s = torch.isnan(s).any(dim=1)
    nan_rows_t = torch.isnan(t).any(dim=1)
    print(f'Rows with NaN in s: {nan_rows_s.sum().item():,}')
    print(f'Rows with NaN in t: {nan_rows_t.sum().item():,}')
    print()

    # Do the NaN rows in X match our known problem pixels?
    print(f'Overlap with ~valid_mask: '
          f'{(nan_rows_s & ~mask).sum().item():,}')
    print(f'NaN rows NOT in ~mask: '
          f'{(nan_rows_s & mask).sum().item():,}')

s (spectral input) stats:
  NaNs: 264630
  Infs: 0
  min : -2.980037
  max : 6.400317

t (topo input) stats:
  NaNs: 52926
  Infs: 0
  min : -8910063172709478931354728238298431488.000000
  max : 7.085165

U1 stats:
  NaNs: 0
  min : -1.341589
  max : 0.575651

U2 stats:
  NaNs: 0
  min : -0.936096
  max : 0.929923

Rows with NaN in s: 52,926
Rows with NaN in t: 52,926

Overlap with ~valid_mask: 52,926
NaN rows NOT in ~mask: 0


In [22]:
with torch.no_grad():
    t = X[:, 5:]
    
    # Find which column contains the extreme value
    print('Per-column min in t:')
    col_names = ['elevation', 'slope', 'aspect_sin', 'aspect_cos',
                 'curvature', 'log_accum', 'panchromatic', 'lwir']
    for i, name in enumerate(col_names):
        col = t[:, i]
        finite = col[torch.isfinite(col)]
        print(f'  {name:<14}: min={col.min().item():>15.4f}  '
              f'max={col.max().item():>12.4f}  '
              f'NaNs={torch.isnan(col).sum().item()}')

Per-column min in t:
  elevation     : min=        -1.2983  max=      3.3008  NaNs=0
  slope         : min=        -1.5524  max=      6.6176  NaNs=0
  aspect_sin    : min=        -1.3858  max=      1.4202  NaNs=0
  aspect_cos    : min=        -1.5944  max=      1.2921  NaNs=0
  curvature     : min=        -2.9208  max=      4.0934  NaNs=0
  log_accum     : min=-8910063172709478931354728238298431488.0000  max=      7.0852  NaNs=0
  panchromatic  : min=        -2.0232  max=      3.3054  NaNs=0
  lwir          : min=            nan  max=         nan  NaNs=52926


In [26]:
# use to determine lambdas

with torch.no_grad():
    A_test = model(X)
    
    LA_elev   = torch.sparse.mm(L_elev_t,   A_test)
    LA_aspect = torch.sparse.mm(L_aspect_t, A_test)
    LA_ddem   = torch.sparse.mm(L_ddem_t,   A_test)
    
    raw_elev   = torch.sum(A_test * LA_elev).item()   / A_test.shape[0]
    raw_aspect = torch.sum(A_test * LA_aspect).item() / A_test.shape[0]
    raw_ddem   = torch.sum(A_test * LA_ddem).item()   / A_test.shape[0]
    raw_band   = model.band_regularization(L_band_tens).item()
    
    A_valid = A_test[mask]
    R_valid = R_tens[mask]
    R_pred  = A_valid @ (E_tens / 21)
    raw_recon = torch.sum((R_valid - R_pred)**2).item() / mask.sum().item()
    
    print('Unweighted loss term magnitudes:')
    print(f'  recon          : {raw_recon:.6f}')
    print(f'  spatial_elev   : {raw_elev:.6f}')
    print(f'  spatial_aspect : {raw_aspect:.6f}')
    print(f'  spatial_ddem   : {raw_ddem:.6f}')
    print(f'  band           : {raw_band:.6f}')
    print()
    print('To balance all terms with recon as reference:')
    print(f'  lam_elev   ~ {raw_recon/raw_elev:.4f}')
    print(f'  lam_aspect ~ {raw_recon/raw_aspect:.4f}')
    print(f'  lam_ddem   ~ {raw_recon/raw_ddem:.4f}')
    print(f'  lam_band   ~ {raw_recon/raw_band:.6f}')

Unweighted loss term magnitudes:
  recon          : 0.094565
  spatial_elev   : 0.094527
  spatial_aspect : 0.144124
  spatial_ddem   : 0.121313
  band           : 6.815396

To balance all terms with recon as reference:
  lam_elev   ~ 1.0004
  lam_aspect ~ 0.6561
  lam_ddem   ~ 0.7795
  lam_band   ~ 0.013875


In [ ]:
# Save trained model weights
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': 100,
    'history': history,
    'hyperparams': {
        'D_s': 5, 'D_t': 7, 'D_out': 32,
        'R': 8, 'D_attn': 16, 'k': 21,
        'lr': 1e-3,
        'lam_elev': 1.0, 'lam_aspect': 0.65,
        'lam_ddem': 0.78, 'lam_band': 0.014
    }
}, 'data/graftn_checkpoint_ep100.pt')

print('saved')